# L0_01 — Leaderboard desde `metrics.json` (Experimentos v3)

Objetivo: construir una tabla comparativa (leaderboard) leyendo los `metrics.json` generados por los experimentos en `data/_experiments/`.

Este notebook no entrena modelos. Sólo agrega resultados ya generados.

## 1) Imports

In [22]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

## 2) Paths

In [23]:
def to_relative(path: Path, root: Path) -> str:
    try:
        return str(path.relative_to(root))
    except ValueError:
        return str(path)


def find_project_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "experimentos_v3").exists():
            return p
    raise FileNotFoundError("No se encontró el root del proyecto (esperado: carpetas 'data' y 'experimentos_v3')")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

EXPERIMENTS_DIR = PROJECT_ROOT / "data" / "_experiments"
METRICS_OUT_DIR = PROJECT_ROOT / "experimentos_final" / "metrics"
METRICS_OUT_DIR.mkdir(parents=True, exist_ok=True)

{
    "experiments_dir": to_relative(EXPERIMENTS_DIR, PROJECT_ROOT),
    "metrics_out_dir": to_relative(METRICS_OUT_DIR, PROJECT_ROOT),
}

{'experiments_dir': 'data/_experiments',
 'metrics_out_dir': 'experimentos_final/metrics'}

## 3) Cargar `metrics.json` y normalizar a tabla

In [24]:
def read_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_record(metrics_path: Path) -> dict:
    payload = read_json(metrics_path)
    meta = payload.get("meta", {})
    holdout = payload.get("holdout", {})

    # Mapeo de nombres de modelos para mayor claridad en el reporte
    model_name_map = {
        "B0_01_naive": "Baseline Naive (Close)",
        "B0_02_naive": "Baseline Naive (OHLCV)",
        "T1_03_transformer_encoder_structured": "Transformer Structured (OHLCV)",
        "T3_01_transformer_gru": "Hybrid Transformer + GRU (Close)",
        "T4_01_transformer_lstm": "Hybrid Transformer + LSTM (Close)",
        "T4_02_transformer_lstm": "Hybrid Transformer + LSTM (OHLCV)",
        "T1_01_transformer_encoder": "Transformer Encoder (Close)",
        "B1_01_ridge": "Ridge Regression (Close)",
        "T2_02_seq2seq_transformer_structured": "Seq2Seq Transformer (OHLCV)"
    }

    raw_model = meta.get("model")
    friendly_model = model_name_map.get(raw_model, raw_model)

    # Filtrar solo los modelos de interés para el reporte final
    models_of_interest = list(model_name_map.keys())
    if raw_model not in models_of_interest:
        return None

    record: dict = {
        "task": meta.get("task"),
        "model": friendly_model,
        "raw_model": raw_model,
        "dataset": meta.get("dataset"),
        "run_id": meta.get("run_id"),
        "n_holdout": holdout.get("n_holdout"),
        "n_epochs_dev": holdout.get("n_epochs_dev"),
        "best_epoch_dev": holdout.get("best_epoch_dev"),
        "metrics_path": to_relative(metrics_path, PROJECT_ROOT),
    }

    if meta.get("task") == "close":
        m = holdout.get("metrics_close", {})
        record.update(
            {
                "holdout_MAE": m.get("MAE"),
                "holdout_RMSE": m.get("RMSE"),
                "holdout_sMAPE": m.get("sMAPE"),
                "holdout_MASE": m.get("MASE"),
            }
        )
        return record

    if meta.get("task") == "ohlcv":
        m = holdout.get("metrics_ohlcv", {})
        record.update(
            {
                "holdout_MAE_OHLC_mean": m.get("MAE_OHLC_mean", m.get("MAE_mean_ohlc")),
                "holdout_RMSE_OHLC_mean": m.get("RMSE_OHLC_mean", m.get("RMSE_mean_ohlc")),
                "holdout_invalid_candle_rate": m.get("invalid_candle_rate", m.get("invalid_candle_rate_pred")),
                "holdout_invalid_volume_rate": m.get("invalid_volume_rate", m.get("invalid_volume_rate_pred")),
            }
        )
        return record

    return record


metrics_paths = sorted(EXPERIMENTS_DIR.rglob("metrics.json"))
records = [normalize_record(p) for p in metrics_paths]
records = [r for r in records if r is not None] # Filter out None records
len(records)

48

In [25]:
metrics_paths = sorted(EXPERIMENTS_DIR.rglob("metrics.json"))
records = [normalize_record(p) for p in metrics_paths]
records = [r for r in records if r is not None] # Filter out None records
df = pd.DataFrame.from_records(records)

df.head()

,task,model,raw_model,dataset,run_id,n_holdout,n_epochs_dev,best_epoch_dev,metrics_path,holdout_MAE,holdout_RMSE,holdout_sMAPE,holdout_MASE,holdout_MAE_OHLC_mean,holdout_RMSE_OHLC_mean,holdout_invalid_candle_rate,holdout_invalid_volume_rate
0,close,Baseline Naive (Close),B0_01_naive,full,20260311_134415,282,NaN,NaN,data/_experiments/close/B0_01_naive/full/20260...,1533.855667,2106.015062,0.016054,2.340206,NaN,NaN,NaN,NaN
1,close,Baseline Naive (Close),B0_01_naive,full,20260311_134500,282,NaN,NaN,data/_experiments/close/B0_01_naive/full/20260...,1533.855667,2106.015062,0.016054,2.340206,NaN,NaN,NaN,NaN
2,close,Baseline Naive (Close),B0_01_naive,full,20260311_135846,282,NaN,NaN,data/_experiments/close/B0_01_naive/full/20260...,1533.855667,2106.015062,0.016054,2.340206,NaN,NaN,NaN,NaN
3,close,Baseline Naive (Close),B0_01_naive,full,20260311_140631,282,NaN,NaN,data/_experiments/close/B0_01_naive/full/20260...,1533.855667,2106.015062,0.016054,2.340206,NaN,NaN,NaN,NaN
4,close,Baseline Naive (Close),B0_01_naive,full,20260311_141029,282,NaN,NaN,data/_experiments/close/B0_01_naive/full/20260...,1533.855667,2106.015062,0.016054,2.340206,NaN,NaN,NaN,NaN


## 4) Leaderboards (holdout)

In [26]:
df_close = df[df["task"] == "close"].copy()
df_ohlcv = df[df["task"] == "ohlcv"].copy()

df_close_sorted = df_close.sort_values(["dataset", "holdout_MAE"], ascending=[True, True])
df_ohlcv_sorted = df_ohlcv.sort_values(["dataset", "holdout_MAE_OHLC_mean"], ascending=[True, True])

cols_close = [
    "dataset",
    "model",
    "run_id",
    "holdout_MAE",
    "holdout_RMSE",
    "holdout_sMAPE",
    "holdout_MASE",
    "n_epochs_dev",
    "metrics_path",
]

cols_ohlcv = [
    "dataset",
    "model",
    "run_id",
    "holdout_MAE_OHLC_mean",
    "holdout_RMSE_OHLC_mean",
    "holdout_invalid_candle_rate",
    "holdout_invalid_volume_rate",
    "n_epochs_dev",
    "metrics_path",
]

df_close_sorted[cols_close].head(30)

,dataset,model,run_id,holdout_MAE,holdout_RMSE,holdout_sMAPE,holdout_MASE,n_epochs_dev,metrics_path
0,full,Baseline Naive (Close),20260311_134415,1533.855667,2106.015062,0.016054,2.340206,NaN,data/_experiments/close/B0_01_naive/full/20260...
1,full,Baseline Naive (Close),20260311_134500,1533.855667,2106.015062,0.016054,2.340206,NaN,data/_experiments/close/B0_01_naive/full/20260...
2,full,Baseline Naive (Close),20260311_135846,1533.855667,2106.015062,0.016054,2.340206,NaN,data/_experiments/close/B0_01_naive/full/20260...
3,full,Baseline Naive (Close),20260311_140631,1533.855667,2106.015062,0.016054,2.340206,NaN,data/_experiments/close/B0_01_naive/full/20260...
4,full,Baseline Naive (Close),20260311_141029,1533.855667,2106.015062,0.016054,2.340206,NaN,data/_experiments/close/B0_01_naive/full/20260...
5,full,Baseline Naive (Close),20260311_141705,1533.855667,2106.015062,0.016054,2.340206,NaN,data/_experiments/close/B0_01_naive/full/20260...
27,full,Hybrid Transformer + LSTM (Close),20260312_021639,1570.034289,2143.874488,0.016394,2.395404,13.0,data/_experiments/close/T4_01_transformer_lstm...
17,full,Transformer Encoder (Close),20260312_013124,1592.523283,2172.122806,0.016637,2.429715,27.0,data/_experiments/close/T1_01_transformer_enco...
21,full,Hybrid Transformer + GRU (Close),20260312_004030,1627.560587,2183.689058,0.016952,2.483172,NaN,data/_experiments/close/T3_01_transformer_gru/...
22,full,Hybrid Transformer + GRU (Close),20260312_010126,1627.560587,2183.689058,0.016952,2.483172,13.0,data/_experiments/close/T3_01_transformer_gru/...


In [27]:
df_ohlcv_sorted[cols_ohlcv].head(30)

,dataset,model,run_id,holdout_MAE_OHLC_mean,holdout_RMSE_OHLC_mean,holdout_invalid_candle_rate,holdout_invalid_volume_rate,n_epochs_dev,metrics_path
42,full,Transformer Structured (OHLCV),20260315_150328,830.775192,1142.031167,0.000000,0.000000,60.0,data/_experiments/ohlcv/T1_03_transformer_enco...
41,full,Transformer Structured (OHLCV),20260312_191638,841.854446,1151.965454,0.000000,0.000000,60.0,data/_experiments/ohlcv/T1_03_transformer_enco...
44,full,Seq2Seq Transformer (OHLCV),20260312_201144,1111.406696,1440.312212,0.000000,0.000000,50.0,data/_experiments/ohlcv/T2_02_seq2seq_transfor...
35,full,Baseline Naive (OHLCV),20260315_150316,1443.975652,2017.915083,0.000000,0.000000,NaN,data/_experiments/ohlcv/B0_02_naive/full/20260...
29,full,Baseline Naive (OHLCV),20260311_201454,1457.710397,2031.229916,0.000000,0.000000,NaN,data/_experiments/ohlcv/B0_02_naive/full/20260...
30,full,Baseline Naive (OHLCV),20260311_203134,1457.710397,2031.229916,0.000000,0.000000,NaN,data/_experiments/ohlcv/B0_02_naive/full/20260...
31,full,Baseline Naive (OHLCV),20260311_204448,1457.710397,2031.229916,0.000000,0.000000,NaN,data/_experiments/ohlcv/B0_02_naive/full/20260...
32,full,Baseline Naive (OHLCV),20260311_205657,1457.710397,2031.229916,0.000000,0.000000,NaN,data/_experiments/ohlcv/B0_02_naive/full/20260...
33,full,Baseline Naive (OHLCV),20260311_210246,1457.710397,2031.229916,0.000000,0.000000,NaN,data/_experiments/ohlcv/B0_02_naive/full/20260...
34,full,Baseline Naive (OHLCV),20260311_211925,1457.710397,2031.229916,0.000000,0.000000,NaN,data/_experiments/ohlcv/B0_02_naive/full/20260...


## 5) Exportar CSV

In [28]:
out_csv = METRICS_OUT_DIR / "leaderboard_1d_holdout_v1.csv"
df.to_csv(out_csv, index=False)
to_relative(out_csv, PROJECT_ROOT)

'experimentos_final/metrics/leaderboard_1d_holdout_v1.csv'